# KubeCon India 2026 — Environment Setup (GCP / KCC track)

The notebook form of [`init-with-kcc.sh`](init-with-kcc.sh) — Track 3: a local cluster
+ KubeVela + **Google Config Connector (KCC)** as the cloud-resource orchestrator for
GCS. Reuses the repo's [`../../../scripts/`](../../../scripts/) helpers.

## Prerequisites
- `k3d`, `kubectl`, `helm`, `vela`, `docker`, `python3`, `curl`, `tar`
- GCP credentials in `gcp-setup/.env.gcp` (project id + path to a service-account JSON key;
  template written on first run if missing)

## Steps
0. Check prerequisites
1. Python venv + load configuration
2. Create the k3d cluster + local registry
3. Establish GCP connectivity (`.env.gcp`)
4. Install the Config Connector operator + a cluster-mode ConfigConnector
5. Install the KubeVela control plane (+ VelaUX)

> After this, run [`01_setup-with-kcc.ipynb`](01_setup-with-kcc.ipynb).


## Step 0: Prerequisites

In [ ]:
# Prerequisites check
import shutil, os, sys, subprocess

print("=== Checking prerequisites ===\n")
tools = ["k3d", "kubectl", "helm", "vela", "docker", "python3", "curl", "tar"]
missing = [t for t in tools if shutil.which(t) is None]
for t in tools:
    print(f"  {'OK ' if shutil.which(t) else 'MISSING'}  {t}")
print("\nconfig.yaml:", "OK" if os.path.exists("config.yaml") else "MISSING")
try:
    import yaml  # noqa: F401
    print("PyYAML:     OK")
except ImportError:
    print("PyYAML:     installing into the kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    print("PyYAML:     OK")
if missing:
    print("\n*** Install the missing tools before continuing:")
    print("    brew install k3d kubectl helm docker python@3.12 curl gnu-tar\ncurl -fsSl https://kubevela.io/script/install.sh | bash   # vela")
elif not os.path.exists("config.yaml"):
    print("\n*** config.yaml is missing from this folder.")
else:
    print("\nAll prerequisites satisfied.")


## Step 1: Python venv + load configuration

Create/activate the demo-local venv (installs PyYAML), then run `load_config` to parse
this folder's `config.yaml`, export the cluster vars, and write `.env.sh` (sourced by
later cells).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/setup-venv.sh"
source "$REPO_ROOT/scripts/load-config.sh"

setup_venv "$PWD/.venv" "$REPO_ROOT/scripts/requirements.txt"
load_config "$PWD/config.yaml"
echo
echo "Config: CLUSTER_NAME=$CLUSTER_NAME  API_PORT=$API_PORT  HTTP_PORT=$HTTP_PORT"


## Step 2: Create the k3d cluster + local registry

`create_cluster` recreates the cluster wired to a local registry, switches the kubectl
context, and verifies access (uses the vars from Step 1).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/create-cluster.sh"

create_cluster


## Step 3: Establish GCP connectivity

`load_gcp_env` sources `gcp-setup/.env.gcp` and exports `GOOGLE_*` (project id +
`GOOGLE_APPLICATION_CREDENTIALS` = path to the SA key). If missing it writes a template
and stops — fill it in, then re-run.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-gcp-env.sh"

load_gcp_env "$PWD/.env.gcp"


## Step 4: Install Config Connector (KCC)

`install_kcc` downloads + applies the Config Connector **operator** release bundle (a
kubectl-applied YAML bundle, not Helm), waits for it, creates the `gcp-key` secret in
`cnrm-system` from the SA key, applies the cluster-mode `ConfigConnector` (from the
versioned `platform/kcc/config-connector/configconnector.yaml`), and waits for
`cnrm-controller-manager`. Re-loads `.env.gcp` first (fresh shell).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-gcp-env.sh"
source "$REPO_ROOT/scripts/create-gcp-secret.sh"
source "$REPO_ROOT/scripts/install-kcc.sh"

load_gcp_env "$PWD/.env.gcp"
install_kcc gcp-key latest "$REPO_ROOT/platform/kcc/config-connector/configconnector.yaml"


## Step 5: Install the KubeVela control plane

`install_kubevela --velaux` installs `vela-core`, enables VelaUX, and port-forwards it
to http://localhost:8000.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/install-kubevela.sh"

install_kubevela --velaux


## Setup complete

The KCC track is bootstrapped: k3d cluster + registry, the Config Connector operator +
controller, and the KubeVela control plane (+ VelaUX). Next:
[`01_setup-with-kcc.ipynb`](01_setup-with-kcc.ipynb).
